# Workstream 6: Retrieval Candidate Diagnostics

source 別・bucket 別に candidate pool が正解 track を含められるかを確認します。まず既存 `exp012_wide_candidate_generation` と `exp005_metadata_retrieval` の artifact を統合し、追加診断の入口にします。

この notebook は Blind A の正解を推測しません。Blind A result dir は artifact 形状確認と source availability のみ扱います。


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Discover Retrieval Artifacts


In [ ]:
retrieval_roots = [
    EXPERIMENT_DIR / "exp012_wide_candidate_generation" / "results",
    EXPERIMENT_DIR / "exp005_metadata_retrieval" / "results",
    EXPERIMENT_DIR / "exp004_bm25_query_views" / "results",
    EXPERIMENT_DIR / "exp007_candidate_union_rerank_gate" / "results",
]

artifact_rows = []
for root in retrieval_roots:
    if not root.exists():
        continue
    for path in sorted(root.rglob("*")):
        if path.suffix in {".csv", ".json", ".jsonl"}:
            artifact_rows.append({
                "experiment": root.parents[0].name,
                "result_group": path.parent.relative_to(root).as_posix(),
                "file": path.name,
                "path": path.relative_to(ROOT).as_posix(),
                "size_kb": round(path.stat().st_size / 1024, 1),
            })
artifact_index = pd.DataFrame(artifact_rows)
show_df(artifact_index, 200)


## Source Recall And Union Recall


In [ ]:
def load_retrieval_csv(file_name: str) -> pd.DataFrame:
    frames = []
    for root in retrieval_roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob(file_name)):
            df = pd.read_csv(path)
            df.insert(0, "artifact", path.relative_to(ROOT).as_posix())
            df.insert(1, "experiment", root.parents[0].name)
            df.insert(2, "result_group", path.parent.relative_to(root).as_posix())
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

source_recall = load_retrieval_csv("source_recall.csv")
union_recall = load_retrieval_csv("union_recall.csv")
source_overlap = load_retrieval_csv("source_overlap.csv")

save_table(source_recall, "retrieval_recall_by_source.csv")
save_table(union_recall, "retrieval_union_recall.csv")
save_table(source_overlap, "source_overlap.csv")

show_df(source_recall, 80)
show_df(union_recall, 80)


## Recall@K Source Comparison


In [ ]:
if not source_recall.empty and {"candidate_k", "recall", "source"}.issubset(source_recall.columns):
    plot_df = source_recall[source_recall["candidate_k"].isin([20, 50, 100, 500])].copy()
    fig, ax = plt.subplots(figsize=(14, 6))
    if sns is not None:
        sns.lineplot(data=plot_df, x="candidate_k", y="recall", hue="source", style="result_group", marker="o", ax=ax)
    ax.set_title("Source recall@K across retrieval result groups")
    fig.tight_layout()
    plt.show()

if not union_recall.empty and {"candidate_k", "recall", "source"}.issubset(union_recall.columns):
    plot_df = union_recall[union_recall["candidate_k"].isin([20, 50, 100, 500])].copy()
    fig, ax = plt.subplots(figsize=(12, 5))
    if sns is not None:
        sns.lineplot(data=plot_df, x="candidate_k", y="recall", hue="result_group", marker="o", ax=ax)
    ax.set_title("Union recall@K")
    fig.tight_layout()
    plt.show()


## Candidate Failure Cases


In [ ]:
failure_rows = []
for root in retrieval_roots:
    if not root.exists():
        continue
    for path in sorted(root.rglob("error_cases.jsonl")):
        count = sum(1 for _ in path.open())
        failure_rows.append({
            "experiment": root.parents[0].name,
            "result_group": path.parent.relative_to(root).as_posix(),
            "failure_cases": count,
            "path": path.relative_to(ROOT).as_posix(),
        })
failure_cases = pd.DataFrame(failure_rows)
save_table(failure_cases, "candidate_failure_cases.csv")
show_df(failure_cases, 100)


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
